# Setup


In [1]:
from optibook.synchronous_client import Exchange

import time
import logging
import pandas as pd

logger = logging.getLogger("client")
logger.setLevel("ERROR")

print("Setup was successful.")

Setup was successful.


# Connecting to the Exchange


In [2]:
# create a connection to the exchange
exchange = Exchange()
_ = exchange.connect()

# Getting market information


In [3]:
# Define a variable storing the instrument_id of the product we are interested in
instrument_id = "NVDA"

### Instrument Details


In [4]:
# Load all instruments (products) available on the exchange
instruments = exchange.get_instruments()
instruments

{'AMD': Instrument(instrument_id=AMD, tick_size=0.1, price_change_limit=PriceChangeLimit(absolute_change=5.0000, relative_change=10.00%), instrument_type=InstrumentType.STOCK, paused=False, expired=False),
 'AVGO': Instrument(instrument_id=AVGO, tick_size=0.1, price_change_limit=PriceChangeLimit(absolute_change=5.0000, relative_change=10.00%), instrument_type=InstrumentType.STOCK, paused=False, expired=False),
 'CRWV': Instrument(instrument_id=CRWV, tick_size=0.1, price_change_limit=PriceChangeLimit(absolute_change=5.0000, relative_change=10.00%), instrument_type=InstrumentType.STOCK, paused=False, expired=False),
 'MU': Instrument(instrument_id=MU, tick_size=0.1, price_change_limit=PriceChangeLimit(absolute_change=5.0000, relative_change=10.00%), instrument_type=InstrumentType.STOCK, paused=False, expired=False),
 'NBIS': Instrument(instrument_id=NBIS, tick_size=0.1, price_change_limit=PriceChangeLimit(absolute_change=5.0000, relative_change=10.00%), instrument_type=InstrumentType.STO

In [5]:
# The returned type of exchange.get_instruments() is a dictionary, select a single instrument
instrument = instruments[instrument_id]
print(instrument)

Instrument(instrument_id=NVDA, tick_size=0.1, price_change_limit=PriceChangeLimit(absolute_change=5.0000, relative_change=10.00%), instrument_type=InstrumentType.STOCK, paused=False, expired=False)


In [6]:
# From any instruments we can get a lot of data:
print(instrument.instrument_id)
print(instrument.instrument_type)
print(instrument.tick_size)

# For particular types of instruments, additional fields are populated (such as e.g. the option expiry, strike and kind, or the future expiry)
# For a STOCK those fields are equal to None
print(instrument.expiry)

print(instrument.option_kind)
print(instrument.strike)
print(instrument.base_instrument_id)

NVDA
InstrumentType.STOCK
0.1
None
None
None
None


In [7]:
# The InstrumentType is an Enum (enumerable) type, that means its value is one of a few supported categories.
# We can use this Enum to compare which specific type of Instrument we are dealing with.

# First import the InstrumentType enum (we would normally do this at the top of the file)
from optibook.common_types import InstrumentType

# Then make a comparison as such:
if instrument.instrument_type == InstrumentType.STOCK:
    print("It is a STOCK.")
elif instrument.instrument_type == InstrumentType.STOCK_OPTION:
    print("It is a STOCK_OPTION.")
elif instrument.instrument_type == InstrumentType.STOCK_FUTURE:
    print("It is a STOCK_FUTURE.")
else:
    print(f"It was another instrument_type: {instrument.instrument_type}.")

It is a STOCK.


### Order Book


In [8]:
# The exchange.get_last_price_book() method returns a large compound datatype storing bids and offers, with price and volumes, and a timestamp.
book = exchange.get_last_price_book(instrument_id)
book

PriceBook(NVDA 2026-11-12 17:10:22.353338+00:00)
 #bids | price | #asks 
       | 141.2 |  1002 
       | 141.1 |  2001 
       | 141.0 |  3000 
       | 140.9 |   44  
       | 140.8 |   10  
  3018 | 140.6 |       
  2027 | 140.5 |       
  1009 | 140.4 |       
   1   |  50.0 |       
   10  |  10.0 |       

In [9]:
# How do we take it apart? First, extract one side of the order book, the bids, say
bids = book.bids
bids

[PriceVolume(price=140.6, volume=3018),
 PriceVolume(price=140.5, volume=2027),
 PriceVolume(price=140.4, volume=1009),
 PriceVolume(price=50.0, volume=1),
 PriceVolume(price=10.0, volume=10)]

In [10]:
# That's a list of PriceVolume combinations, which makes sense, one line of the order book is always a price with corresponding volume and we have multiple such orders
# Let's find the most competitive order, which is always the first entry in the list
best_bid = bids[0]
best_bid

PriceVolume(price=140.6, volume=3018)

In [11]:
# That's still a compound of price and volume, we can simply select which one we want to see to finally end up with a numeric type
print(best_bid.price)
print(best_bid.volume)

140.6
3018


In [12]:
# Putting it all together we can also write this.
book.bids[0].price

140.6

In [13]:
# Note you will see an error on the above line if there are no bids at all available (the list will have no entry 0)
# Better is to check first
if book.bids:
    print(book.bids[0].price)
else:
    print("No bids in the order book at all.")

140.6


## Orders


### Inserting


In [14]:
# Insert bid LIMIT order - this trades against any current orders, and any remainders become new resting orders in the book
# The returned value is a 'InsertOrderResponse' which, if successfull, contains the order_id of the order
exchange.insert_order(instrument_id, price=10, volume=5, side="bid", order_type="limit")

InsertOrderResponse(success=True, order_id=1843932, error_reason='None')

In [15]:
# Insert ask LIMIT order - this trades against any current orders, and any remainders become new resting orders in the book
exchange.insert_order(instrument_id, price=20, volume=5, side="ask", order_type="limit")

InsertOrderResponse(success=True, order_id=1843937, error_reason='None')

In [16]:
# Insert bid IOC order - this order trades against any resting volume which has an equal or better price, but does not remain in the
# book if any volume is unfilled
exchange.insert_order(instrument_id, price=10, volume=5, side="bid", order_type="ioc")

InsertOrderResponse(success=True, order_id=1843948, error_reason='None')

In [17]:
# Insert ask IOC order - this order trades against any resting volume which has an equal or better price, but does not remain in the
# book if any volume is unfilled
exchange.insert_order(instrument_id, price=20, volume=5, side="ask", order_type="ioc")

InsertOrderResponse(success=True, order_id=1843952, error_reason='None')

### Deleting & amending


In [18]:
# Load a list of own outstanding/resting orders
exchange.get_outstanding_orders(instrument_id)

{1843932: OrderStatus(order_id=1843932, instrument_id=NVDA, price=10.0, volume=5, side=bid)}

In [19]:
# Delete an order by order_id, the returned DeleteOrderRequest contains a boolean showing whether the delete was succesful
exchange.delete_order(instrument_id, order_id=1234)

order delete failed with reason: Could not find order id to delete


DeleteOrderResponse(success=False, error_reason='Could not find order id to delete')

In [20]:
# Delete all outstanding orders for an instrument
exchange.delete_orders(instrument_id)

In [21]:
# Change (amend) the volume of an existing resting order by order_id, the returned AmendOrderResponse contains a boolean showing whether the amend was succesful
exchange.amend_order(instrument_id, order_id=1234, volume=30)

order amend failed with reason: Could not find order id to amend


AmendOrderResponse(success=False, error_reason='Could not find order id to amend')

### Public tradeticks


In [22]:
# Load and store a list of all public tradeticks since the instantiation of the Exchange (upto a max limit)
# These are trades between any participant, you may or may not have participated in these
trade_tick_history = exchange.get_trade_tick_history(instrument_id)

# Display last 5
trade_tick_history[-5:]

[TradeTick(timestamp=2026-11-12 17:10:21.962269+00:00, instrument_id=NVDA, price=140.70000000000002, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=162987),
 TradeTick(timestamp=2026-11-12 17:10:22.282523+00:00, instrument_id=NVDA, price=140.6, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=162988),
 TradeTick(timestamp=2026-11-12 17:10:22.691563+00:00, instrument_id=NVDA, price=140.8, volume=10, aggressor_side=bid, buyer=, seller=, trade_id=162995),
 TradeTick(timestamp=2026-11-12 17:10:22.818092+00:00, instrument_id=NVDA, price=140.6, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=162996),
 TradeTick(timestamp=2026-11-12 17:10:22.933819+00:00, instrument_id=NVDA, price=140.6, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=162997)]

In [23]:
# Poll all public tradeticks since the last time this method was called
exchange.poll_new_trade_ticks(instrument_id)

[TradeTick(timestamp=2026-11-12 17:10:21.962245+00:00, instrument_id=NVDA, price=140.70000000000002, volume=10, aggressor_side=ask, buyer=, seller=, trade_id=162986),
 TradeTick(timestamp=2026-11-12 17:10:21.962269+00:00, instrument_id=NVDA, price=140.70000000000002, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=162987),
 TradeTick(timestamp=2026-11-12 17:10:22.282523+00:00, instrument_id=NVDA, price=140.6, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=162988),
 TradeTick(timestamp=2026-11-12 17:10:22.691563+00:00, instrument_id=NVDA, price=140.8, volume=10, aggressor_side=bid, buyer=, seller=, trade_id=162995),
 TradeTick(timestamp=2026-11-12 17:10:22.818092+00:00, instrument_id=NVDA, price=140.6, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=162996),
 TradeTick(timestamp=2026-11-12 17:10:22.933819+00:00, instrument_id=NVDA, price=140.6, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=162997)]

### Private trades


In [24]:
# Load and store a list of all private trades you participated in since the instantiation of the Exchange (upto a max limit)
trade_history = exchange.get_trade_history(instrument_id)

# Display last 5
trade_history[-5:]

[Trade(timestamp=2026-11-12 17:10:22.818051+00:00, order_id=1843937, trade_id=162996, instrument_id=NVDA, price=140.6, volume=5, side=ask),
 Trade(timestamp=2026-11-12 17:10:22.933783+00:00, order_id=1843952, trade_id=162997, instrument_id=NVDA, price=140.6, volume=5, side=ask)]

In [25]:
# Poll all private trades since the last time this method was called
exchange.poll_new_trades(instrument_id)

[Trade(timestamp=2026-11-12 17:10:22.818051+00:00, order_id=1843937, trade_id=162996, instrument_id=NVDA, price=140.6, volume=5, side=ask),
 Trade(timestamp=2026-11-12 17:10:22.933783+00:00, order_id=1843952, trade_id=162997, instrument_id=NVDA, price=140.6, volume=5, side=ask)]

### Position & PnL


In [26]:
# Load current positions in all instruments
exchange.get_positions()

{'NVDA': -10,
 'MU': 0,
 'CRWV': 0,
 'VRT': 0,
 'SNDK': 0,
 'AVGO': 0,
 'NBIS': 0,
 'AMD': 0}

In [27]:
# Load current positions in all instruments, including cash component (how much did we invest or gain by trading each instrument)
exchange.get_positions_and_cash()

{'NVDA': {'volume': -10, 'cash': 1406.0},
 'MU': {'volume': 0, 'cash': 0.0},
 'CRWV': {'volume': 0, 'cash': 0.0},
 'VRT': {'volume': 0, 'cash': 0.0},
 'SNDK': {'volume': 0, 'cash': 0.0},
 'AVGO': {'volume': 0, 'cash': 0.0},
 'NBIS': {'volume': 0, 'cash': 0.0},
 'AMD': {'volume': 0, 'cash': 0.0}}

In [28]:
# Current PnL: sum of all cash components and value of the positions (evaluated against last-traded price)
exchange.get_pnl()

0.0

# Historical Data


The `HistoricalDataLoader` downloads past trades, orders and order book snapshots (pricebooks) as pandas DataFrames or as the same typed objects the `Exchange` client returns.

Downloaded files are cached locally, so repeated calls for the same period are fast.


In [29]:
from datetime import datetime, timedelta
from optibook.historical_data_loader import HistoricalDataLoader

# Host and credentials picked up from the config file
loader = HistoricalDataLoader()

### Trades


In [30]:
# Pick a time range. Loading is limited to 1 day per call (see "Larger time ranges" below).
end = datetime.utcnow()
start = end - timedelta(hours=1)

# Download and load all public trades in the range as a DataFrame
# Columns: timestamp_ns, instrument_id, price, volume, aggressor_side, buyer, seller, trade_id
trades_df = loader.get_trades_df(start, end)
trades_df.tail()

/tmp/ipykernel_13518/2128448903.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


,timestamp_ns,timestamp_str,instrument_id,price,volume,aggressor_side,buyer,seller,trade_id,exchange_started_at
6166,1794503339476393763,2026-11-12T17:08:59.476394+00:00,NVDA,142.2,2,bid,,,162745,2026-11-11T12:28:39.801042+00:00
6167,1794503339476393763,2026-11-12T17:08:59.476394+00:00,NVDA,142.2,2,bid,,,162746,2026-11-11T12:28:39.801042+00:00
6168,1794503339476393763,2026-11-12T17:08:59.476394+00:00,NVDA,142.2,6,bid,,,162747,2026-11-11T12:28:39.801042+00:00
6169,1794503339809212980,2026-11-12T17:08:59.809213+00:00,NVDA,142.1,10,ask,,,162748,2026-11-11T12:28:39.801042+00:00
6170,1794503339809212980,2026-11-12T17:08:59.809213+00:00,NVDA,142.1,5,ask,,,162749,2026-11-11T12:28:39.801042+00:00


In [31]:
# The timestamp is in nanoseconds since the epoch, convert it to a datetime column for plotting or grouping
trades_df["timestamp"] = pd.to_datetime(trades_df["timestamp_ns"], unit="ns", utc=True)

# Traded volume per instrument
trades_df.groupby("instrument_id")["volume"].sum()

instrument_id
AMD     12405
AVGO    12611
CRWV    13105
MU      13518
NBIS    13269
NVDA    12769
SNDK    12656
VRT     13530
Name: volume, dtype: int32

In [32]:
# Trades for a single instrument, and its volume-weighted average price over the range
instrument_trades = trades_df[trades_df["instrument_id"] == instrument_id]

if not instrument_trades.empty:
    vwap = (instrument_trades["price"] * instrument_trades["volume"]).sum() / instrument_trades["volume"].sum()
    print(f"{instrument_id}: {len(instrument_trades)} trades, VWAP {vwap:.2f}")
else:
    print(f"No trades in {instrument_id} in this range.")

NVDA: 853 trades, VWAP 146.74


In [33]:
# The same data as a list of TradeTick objects, identical to what exchange.get_trade_tick_history() returns
trades = loader.get_trades(start, end)

# Display last 5
trades[-5:]

[TradeTick(timestamp=2026-11-12 17:08:59.476394+00:00, instrument_id=NVDA, price=142.20000000000002, volume=2, aggressor_side=bid, buyer=, seller=, trade_id=162745),
 TradeTick(timestamp=2026-11-12 17:08:59.476394+00:00, instrument_id=NVDA, price=142.20000000000002, volume=2, aggressor_side=bid, buyer=, seller=, trade_id=162746),
 TradeTick(timestamp=2026-11-12 17:08:59.476394+00:00, instrument_id=NVDA, price=142.20000000000002, volume=6, aggressor_side=bid, buyer=, seller=, trade_id=162747),
 TradeTick(timestamp=2026-11-12 17:08:59.809213+00:00, instrument_id=NVDA, price=142.1, volume=10, aggressor_side=ask, buyer=, seller=, trade_id=162748),
 TradeTick(timestamp=2026-11-12 17:08:59.809213+00:00, instrument_id=NVDA, price=142.1, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=162749)]

### Orders


In [34]:
# All order book updates in the range as a DataFrame
# Columns: timestamp_ns, instrument_id, order_id, price, volume, side, username
orders_df = loader.get_orders_df(start, end)
orders_df.tail()

,timestamp_ns,timestamp_str,instrument_id,order_id,price,volume,side,username,action,order_type,exchange_started_at
152079,1794503339867275204,2026-11-12T17:08:59.867275+00:00,VRT,1839117,69.6,2,ask,,new,limit,2026-11-11T12:28:39.801042+00:00
152080,1794503339869002357,2026-11-12T17:08:59.869002+00:00,VRT,1839118,69.1,2,bid,,new,limit,2026-11-11T12:28:39.801042+00:00
152081,1794503339971516913,2026-11-12T17:08:59.971517+00:00,NVDA,1839092,142.3,0,ask,,cancelled,limit,2026-11-11T12:28:39.801042+00:00
152082,1794503339973255343,2026-11-12T17:08:59.973255+00:00,NVDA,1839119,142.0,10,bid,,new,limit,2026-11-11T12:28:39.801042+00:00
152083,1794503339974850699,2026-11-12T17:08:59.974851+00:00,NVDA,1839120,142.2,10,ask,,new,limit,2026-11-11T12:28:39.801042+00:00


In [35]:
# Only the bid orders on our instrument
bids_df = orders_df[(orders_df["instrument_id"] == instrument_id) & (orders_df["side"] == "bid")]
bids_df.tail()

,timestamp_ns,timestamp_str,instrument_id,order_id,price,volume,side,username,action,order_type,exchange_started_at
152053,1794503339809212980,2026-11-12T17:08:59.809213+00:00,NVDA,1839101,142.1,0,bid,,fill,limit,2026-11-11T12:28:39.801042+00:00
152060,1794503339839942758,2026-11-12T17:08:59.839943+00:00,NVDA,1839107,142.0,5,bid,,new,limit,2026-11-11T12:28:39.801042+00:00
152070,1794503339855039354,2026-11-12T17:08:59.855039+00:00,NVDA,1839057,141.7,0,bid,,cancelled,limit,2026-11-11T12:28:39.801042+00:00
152072,1794503339858463165,2026-11-12T17:08:59.858463+00:00,NVDA,1839114,141.9,2,bid,,new,limit,2026-11-11T12:28:39.801042+00:00
152082,1794503339973255343,2026-11-12T17:08:59.973255+00:00,NVDA,1839119,142.0,10,bid,,new,limit,2026-11-11T12:28:39.801042+00:00


### Pricebooks


In [36]:
# Order book snapshots in the range as a DataFrame
# Columns: timestamp_ns, instrument_id, bids_json, asks_json (the two sides are JSON strings of [price, volume] pairs)
pricebooks_df = loader.get_pricebooks_df(start, end)
pricebooks_df.tail()

,timestamp_ns,timestamp_str,instrument_id,bids_json,asks_json,exchange_started_at
134708,1794503339867313449,2026-11-12T17:08:59.867313+00:00,VRT,"[[69.2, 3015], [69.10000000000001, 2000], [69....","[[69.5, 3013], [69.60000000000001, 2003], [69....",2026-11-11T12:28:39.801042+00:00
134709,1794503339869030083,2026-11-12T17:08:59.869030+00:00,VRT,"[[69.2, 3015], [69.10000000000001, 2002], [69....","[[69.5, 3013], [69.60000000000001, 2003], [69....",2026-11-11T12:28:39.801042+00:00
134710,1794503339971529158,2026-11-12T17:08:59.971529+00:00,NVDA,"[[142.0, 8], [141.9, 16], [141.8, 3000], [141....","[[142.20000000000002, 3018], [142.3, 2003], [1...",2026-11-11T12:28:39.801042+00:00
134711,1794503339973288790,2026-11-12T17:08:59.973289+00:00,NVDA,"[[142.0, 18], [141.9, 16], [141.8, 3000], [141...","[[142.20000000000002, 3018], [142.3, 2003], [1...",2026-11-11T12:28:39.801042+00:00
134712,1794503339974885343,2026-11-12T17:08:59.974885+00:00,NVDA,"[[142.0, 18], [141.9, 16], [141.8, 3000], [141...","[[142.20000000000002, 3028], [142.3, 2003], [1...",2026-11-11T12:28:39.801042+00:00


In [37]:
# As PriceBook objects, identical to what exchange.get_last_price_book() returns
pricebooks = loader.get_pricebooks(start, end)

# Best bid and ask of the most recent snapshot of our instrument
for book in reversed(pricebooks):
    if book.instrument_id == instrument_id:
        best_bid = book.bids[0].price if book.bids else None
        best_ask = book.asks[0].price if book.asks else None
        print(f"{book.timestamp}: best bid {best_bid}, best ask {best_ask}")
        break
else:
    print(f"No pricebook snapshots for {instrument_id} in this range.")

2026-11-12 17:08:59.974885+00:00: best bid 142.0, best ask 142.20000000000002


### Larger time ranges


Loading is limited to 1 day per call to protect memory. For longer periods, download everything first (no limit), then load and process one day at a time from the local cache.


In [ ]:
week_end = datetime.utcnow()
week_start = week_end - timedelta(days=7)

# Download all feeds (trades, orders, pricebooks) for the whole week. Already-cached files are skipped.
loader.download_all(week_start, week_end)

# Load from the cache one day at a time
day_start = week_start
while day_start < week_end:
    day_end = min(day_start + timedelta(days=1), week_end)
    day_trades = loader.load_trades_df(day_start, day_end)
    print(f"{day_start:%Y-%m-%d}: {len(day_trades)} trades")
    day_start = day_end

/tmp/ipykernel_13518/2157710028.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  week_end = datetime.utcnow()


# Code Snippets


Combining a few of the exchange interactions above, we can write a code snippet to aggressively trade out of all currently held positions using IOC orders.

That means selling all products you are long, and buying back all you are short.

This is useful for starting with a clean slate, but doing so will come at a potentially large cost.

You are not guaranteed to end up with a zero position afterwards, only that volume which is also available to trade in the market will be bought/sold.


In [ ]:
MIN_SELLING_PRICE = 0.10
MAX_BUYING_PRICE = 100000.00

positions = exchange.get_positions()
pnl = exchange.get_pnl()

print(f"Positions before: {positions}")
print(f"\nPnL before: {pnl:.2f}")

print(f"\nTrading out of positions")
for iid, pos in positions.items():
    if pos > 0:
        print(
            f"-- Inserting sell order for {pos} lots of {iid}, with limit price {MIN_SELLING_PRICE:.2f}"
        )
        exchange.insert_order(
            iid, price=MIN_SELLING_PRICE, volume=pos, side="ask", order_type="ioc"
        )
    elif pos < 0:
        print(
            f"-- Inserting buy order for {abs(pos)} lots of {iid}, with limit price {MAX_BUYING_PRICE:.2f}"
        )
        exchange.insert_order(
            iid, price=MAX_BUYING_PRICE, volume=-pos, side="bid", order_type="ioc"
        )
    else:
        print(f"-- No initial position in {iid}, skipping..")

    time.sleep(0.10)

time.sleep(1.0)

positions = exchange.get_positions()
pnl = exchange.get_pnl()
print(f"\nPositions after: {positions}")
print(f"\nPnL after: {pnl:.2f}")